In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# CROP YIELD DATASET - PRODUCTION QUALITY CLEANING
# ============================================================

INPUT_FILE = "crop_yield_dataset_india.csv"
OUTPUT_FILE = "crop_yield_dataset_india_clean.csv"

# Create output directory
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("CROP YIELD DATASET CLEANING")
print("=" * 60)

print("Original shape:", df.shape)

# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

required_columns = [
    "country",
    "crop",
    "year",
    "area_harvested_ha",
    "production_tonnes",
    "yield_hg_per_ha"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Keep expected columns
df = df[required_columns].copy()

# ------------------------------------------------------------
# 3. Clean text columns
# ------------------------------------------------------------

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
)

df["crop"] = (
    df["crop"]
    .astype("string")
    .str.strip()
)

# Standardize country representation
df["country"] = df["country"].replace({
    "IND": "India",
    "INDIA": "India",
    "india": "India"
})

# ------------------------------------------------------------
# 4. Convert numerical columns
# ------------------------------------------------------------

numeric_columns = [
    "year",
    "area_harvested_ha",
    "production_tonnes",
    "yield_hg_per_ha"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 5. Validate year
# ------------------------------------------------------------

# Expected agricultural data period:
# 2010 to 2024 based on this dataset.

invalid_year = ~df["year"].between(
    2010,
    2024
)

df.loc[
    invalid_year,
    "year"
] = pd.NA

# ------------------------------------------------------------
# 6. Validate area harvested
# ------------------------------------------------------------

invalid_area = (
    df["area_harvested_ha"] < 0
)

df.loc[
    invalid_area,
    "area_harvested_ha"
] = pd.NA

# ------------------------------------------------------------
# 7. Validate production
# ------------------------------------------------------------

invalid_production = (
    df["production_tonnes"] < 0
)

df.loc[
    invalid_production,
    "production_tonnes"
] = pd.NA

# ------------------------------------------------------------
# 8. Validate yield
# ------------------------------------------------------------

invalid_yield = (
    df["yield_hg_per_ha"] < 0
)

df.loc[
    invalid_yield,
    "yield_hg_per_ha"
] = pd.NA

# ------------------------------------------------------------
# 9. Remove exact duplicate records
# ------------------------------------------------------------

rows_before_duplicates = len(df)

df = df.drop_duplicates()

df = df.reset_index(drop=True)

duplicates_removed = (
    rows_before_duplicates - len(df)
)

# ------------------------------------------------------------
# 10. Validate production-area-yield relationship
# ------------------------------------------------------------

# FAOSTAT-style yield:
#
# yield (hg/ha) =
# production (tonnes) * 100000
# --------------------------------
# area (ha)
#
# We DO NOT overwrite the source yield.
# We only report large inconsistencies.

valid_calculation = (
    df["area_harvested_ha"].notna()
    & df["production_tonnes"].notna()
    & df["yield_hg_per_ha"].notna()
    & (df["area_harvested_ha"] > 0)
)

calculated_yield = (
    df.loc[valid_calculation, "production_tonnes"]
    * 100000
    / df.loc[valid_calculation, "area_harvested_ha"]
)

reported_yield = (
    df.loc[valid_calculation, "yield_hg_per_ha"]
)

relative_difference = (
    (calculated_yield - reported_yield).abs()
    / reported_yield.replace(0, pd.NA)
)

inconsistent_yield_count = (
    relative_difference > 0.05
).sum()

# ------------------------------------------------------------
# 11. Final validation report
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(
    "Rows before cleaning:",
    rows_before_duplicates
)

print(
    "Rows after cleaning:",
    len(df)
)

print(
    "Duplicate rows removed:",
    duplicates_removed
)

print(
    "Invalid year values:",
    invalid_year.sum()
)

print(
    "Invalid area values:",
    invalid_area.sum()
)

print(
    "Invalid production values:",
    invalid_production.sum()
)

print(
    "Invalid yield values:",
    invalid_yield.sum()
)

print(
    "Yield consistency warnings:",
    inconsistent_yield_count
)

print("\nMissing values:")

print(
    df.isnull().sum()
)

print("\nData types:")

print(
    df.dtypes
)

# ------------------------------------------------------------
# 12. Save cleaned dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 60)
print("CLEANING COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    "Cleaned file saved to:",
    OUTPUT_FILE
)

CROP YIELD DATASET CLEANING
Original shape: (2361, 6)

CLEANING SUMMARY
Rows before cleaning: 2361
Rows after cleaning: 2361
Duplicate rows removed: 0
Invalid year values: 0
Invalid area values: 0
Invalid production values: 0
Invalid yield values: 0
Yield consistency warnings: 1346

Missing values:
country                 0
crop                    0
year                    0
area_harvested_ha    1015
production_tonnes       0
yield_hg_per_ha      1000
dtype: int64

Data types:
country              string[python]
crop                 string[python]
year                        float64
area_harvested_ha           float64
production_tonnes           float64
yield_hg_per_ha             float64
dtype: object

CLEANING COMPLETED SUCCESSFULLY
Cleaned file saved to: crop_yield_dataset_india_clean.csv
